# Sprint 2 — Ingeniería de Targets y Split Temporal (v3)
## Proyecto: Productividad Asesores de Negocios

---

### Cambios respecto a v2

Las variables de evento (CLIENTES_PRESTAMO, PRESTAMO, CLIENTES_NUEVOS, DESEMBOLSO_CLIENTES_NUEVOS) se calculaban promediando sobre todas las semanas, incluyendo semanas sin desembolso (valor cero). Eso distorsiona su significado.

En v3 se reemplazan por 4 variables calculadas **solo sobre semanas con evento**:

| Variable nueva | Calculo | Interpretacion |
|---------------|---------|----------------|
| `TASA_DESEMBOLSO` | filas con evento / total filas | Actividad de renovacion de cartera |
| `CLIENTES_PRESTAMO_EVENTO` | mean(CLIENTES_PRESTAMO) donde > 0 | Tamano promedio del grupo al desembolso |
| `MONTO_DESEMBOLSO_EVENTO` | mean(PRESTAMO) donde > 0 | Monto promedio desembolsado por grupo |
| `CLIENTES_NUEVOS_EVENTO` | mean(CLIENTES_NUEVOS) donde > 0 | Clientes nuevos promedio en desembolsos |

### Evidencia que justifica el cambio

| Variable | BAJO | MEDIO | ALTO | Discrimina |
|----------|------|-------|------|------------|
| CLIENTES_PRESTAMO (promedio total) | 0.90 | 0.59 | 0.33 | Si (pero por frecuencia, no tamano) |
| CLIENTES_PRESTAMO_EVENTO | 10.20 | 10.04 | 10.08 | No (tamano similar entre clases) |
| PRESTAMO_EVENTO | $127,919 | $122,774 | $107,590 | Si |
| TASA_DESEMBOLSO | 15.9% | 6.6% | 3.6% | Si (1 de cada 6 vs 1 de cada 28 filas) |

**Interpretacion de TASA_DESEMBOLSO:**
El credito grupal estandar es 12 semanas. Un asesor BAJO renueva activamente (1 de cada 6 filas-grupo con desembolso). Un asesor ALTO tiene grupos estancados que no califican para renovacion (1 de cada 28).


## 1. Configuracion y carga de datos

In [1]:
import warnings
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

DB_PATH    = Path('../data/advisor_risk.duckdb')
INTERIM    = Path('../data/interim')
FIG_DIR    = Path('../reports/figures')
TABLE_NAME = 'tabla_hist_asesores'

INTERIM.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect(str(DB_PATH), read_only=True)
df   = conn.execute(f'SELECT * FROM {TABLE_NAME}').df()
conn.close()

for col in ['HOY', 'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Dataset cargado: {len(df):,} filas')
print(f'Periodo: {df["HOY"].min().date()} -> {df["HOY"].max().date()}')
print(f'Asesores unicos: {df["CODIGO_ASESOR"].nunique():,}')
print(f'Filas con CLIENTES_PRESTAMO > 0: {(df["CLIENTES_PRESTAMO"] > 0).sum():,} ({(df["CLIENTES_PRESTAMO"] > 0).mean()*100:.1f}%)')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset cargado: 1,218,831 filas
Periodo: 2016-06-22 -> 2019-01-30
Asesores unicos: 2,004
Filas con CLIENTES_PRESTAMO > 0: 70,437 (5.8%)


## 2. Construccion de variables de evento

Calculamos las 4 variables nuevas antes del colapso a nivel asesor.


In [2]:
def calcular_vars_evento(df):
    '''
    Calcula variables de desembolso correctamente:
    - Solo sobre filas donde hubo desembolso (CLIENTES_PRESTAMO > 0)
    - TASA_DESEMBOLSO: filas con evento / total filas por asesor
    '''
    # Subconjunto de filas con evento de desembolso
    df_evento = df[df['CLIENTES_PRESTAMO'] > 0]

    # Total de filas por asesor (denominador de la tasa)
    total_filas = df.groupby('CODIGO_ASESOR').size().rename('_total_filas')

    # Filas con evento por asesor
    filas_evento = df_evento.groupby('CODIGO_ASESOR').size().rename('_filas_evento')

    # TASA_DESEMBOLSO
    tasa = (filas_evento / total_filas).fillna(0).rename('TASA_DESEMBOLSO')

    # Variables de evento: media solo sobre filas con desembolso
    clientes_evento = (
        df_evento.groupby('CODIGO_ASESOR')['CLIENTES_PRESTAMO']
        .mean().rename('CLIENTES_PRESTAMO_EVENTO')
    )
    monto_evento = (
        df_evento.groupby('CODIGO_ASESOR')['PRESTAMO']
        .mean().rename('MONTO_DESEMBOLSO_EVENTO')
    )
    nuevos_evento = (
        df_evento.groupby('CODIGO_ASESOR')['CLIENTES_NUEVOS']
        .mean().rename('CLIENTES_NUEVOS_EVENTO')
    )

    return pd.concat([
        tasa, clientes_evento, monto_evento, nuevos_evento
    ], axis=1)


vars_evento = calcular_vars_evento(df)
print(f'Variables de evento calculadas: {vars_evento.shape}')
print(vars_evento.describe().round(2))


Variables de evento calculadas: (2004, 4)
       TASA_DESEMBOLSO  CLIENTES_PRESTAMO_EVENTO  MONTO_DESEMBOLSO_EVENTO  \
count       2,004.0000                1,852.0000               1,852.0000   
mean            0.0900                   10.0400             110,971.6000   
std             0.1200                    1.3000              43,296.8800   
min             0.0000                    3.3600              24,000.0000   
25%             0.0400                    9.2000              80,702.3800   
50%             0.0600                   10.0000             104,068.8900   
75%             0.0900                   10.7500             133,898.8100   
max             1.0000                   18.0000             456,000.0000   

       CLIENTES_NUEVOS_EVENTO  
count              1,852.0000  
mean                   3.0300  
std                    2.1000  
min                    0.0000  
25%                    1.6200  
50%                    2.4600  
75%                    3.8400  
max     

## 3. Colapso a nivel asesor

In [3]:
def build_target_b_v3(df, vars_evento):
    '''
    Colapsa el dataset a nivel asesor con:
    - 4 variables de evento correctamente calculadas
    - Target de 3 clases (terciles de TDNC)
    - Variables originales de evento EXCLUIDAS
    '''
    print('Colapsando dataset a nivel asesor...')

    # Variables promediadas sobre TODAS las semanas
    # (excluimos las 4 originales de evento que se reemplazan)
    AGG_MEDIA = [
        'GRUPOS', 'CLIENTES',
        'CARTERA_HOY', 'CARTERA_SEMANT', 'INCREMENTO_CARTERA',
        'ATRASO', 'TASA_PROM', 'PROVISION_HOY', 'PROVISION_SEMANT',
        'GASTO_PROVISION_SEMANAL', 'DIAS_MORA',
        # EXCLUIDAS: PRESTAMO, CLIENTES_PRESTAMO, CLIENTES_NUEVOS,
        # DESEMBOLSO_CLIENTES_NUEVOS (reemplazadas por vars_evento)
    ]
    AGG_ULTIMA = [
        'REGION', 'SUCURSAL', 'CODIGO_SUCURSAL', 'COORDINADOR',
        'PUESTO', 'PRODUCTO', 'AREA', 'SEXO', 'ESTADO_CIVIL',
        'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'EDAD',
        'RANGO_CICLO', 'DIA_PAGO', 'TIPO_BAJA', 'MOTIVO_BAJA',
        'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO',
    ]

    agg_media   = df.groupby('CODIGO_ASESOR')[AGG_MEDIA].mean()
    agg_ultima  = df.groupby('CODIGO_ASESOR')[AGG_ULTIMA].last()
    agg_semanas = df.groupby('CODIGO_ASESOR')['HOY'].nunique().rename('N_SEMANAS_OBS')
    agg_p_fecha = df.groupby('CODIGO_ASESOR')['HOY'].min().rename('PRIMERA_OBS')
    agg_u_fecha = df.groupby('CODIGO_ASESOR')['HOY'].max().rename('ULTIMA_OBS')

    df_asesor = pd.concat([
        agg_media, agg_ultima, agg_semanas,
        agg_p_fecha, agg_u_fecha, vars_evento
    ], axis=1).reset_index()

    # TDNC para construir el target
    epsilon = 1e-6
    df_asesor['TDNC'] = (
        df_asesor['ATRASO'] / (df_asesor['CARTERA_HOY'] + epsilon)
    ).clip(lower=0)

    # TARGET: 3 clases (terciles de TDNC)
    df_asesor['TARGET_B'] = pd.qcut(
        df_asesor['TDNC'],
        q=3,
        labels=['BAJO', 'MEDIO', 'ALTO'],
        duplicates='drop',
    )

    print(f'Dataset colapsado: {len(df_asesor):,} asesores')
    return df_asesor


df_asesores = build_target_b_v3(df, vars_evento)

print('\n=== DISTRIBUCION TARGET_B (3 clases) ===')
dist = df_asesores['TARGET_B'].value_counts(dropna=False).reset_index()
dist.columns = ['clase', 'asesores']
dist['porcentaje'] = (dist['asesores'] / len(df_asesores) * 100).round(2)
display(dist)

print('\n=== TDNC POR CLASE ===')
display(df_asesores.groupby('TARGET_B')['TDNC'].describe().round(4))

print('\n=== VERIFICACION VARIABLES DE EVENTO POR CLASE ===')
vars_check = [
    'TASA_DESEMBOLSO', 'CLIENTES_PRESTAMO_EVENTO',
    'MONTO_DESEMBOLSO_EVENTO', 'CLIENTES_NUEVOS_EVENTO'
]
display(df_asesores.groupby('TARGET_B')[vars_check].mean().round(4))

# Imputar NaN en variables de evento con 0
# Justificacion: NaN significa que el asesor nunca tuvo un desembolso
# por lo tanto el promedio de sus grupos al desembolso es cero
vars_evento_cols = [
    'TASA_DESEMBOLSO', 'CLIENTES_PRESTAMO_EVENTO',
    'MONTO_DESEMBOLSO_EVENTO', 'CLIENTES_NUEVOS_EVENTO'
]
for col in vars_evento_cols:
    n_nulos = df_asesores[col].isna().sum()
    if n_nulos > 0:
        df_asesores[col] = df_asesores[col].fillna(0)
        print(f"  {col}: {n_nulos} NaN imputados con 0")

print(f"\nNaN restantes en variables de evento: {df_asesores[vars_evento_cols].isna().sum().sum()}")

# Verificar que la imputacion no distorsiona las medias por clase
print("\n=== VERIFICACION FINAL VARIABLES DE EVENTO POR CLASE ===")
display(df_asesores.groupby('TARGET_B')[vars_evento_cols].mean().round(4))

print('\nInterpretacion TASA_DESEMBOLSO:')
for clase in ['BAJO', 'MEDIO', 'ALTO']:
    tasa = df_asesores[df_asesores['TARGET_B']==clase]['TASA_DESEMBOLSO'].mean()
    equiv = 1/tasa if tasa > 0 else float('inf')
    print(f"  {clase:<8}: {tasa:.4f} = 1 de cada {equiv:.0f} filas-grupo con desembolso")

print('\nInterpretacion TASA_DESEMBOLSO:')
for clase in ['BAJO', 'MEDIO', 'ALTO']:
    tasa = df_asesores[df_asesores['TARGET_B']==clase]['TASA_DESEMBOLSO'].mean()
    equiv = 1/tasa if tasa > 0 else float('inf')
    print(f'  {clase:<8}: {tasa:.4f} = 1 de cada {equiv:.0f} filas-grupo con desembolso')





Colapsando dataset a nivel asesor...
Dataset colapsado: 2,004 asesores

=== DISTRIBUCION TARGET_B (3 clases) ===


,clase,asesores,porcentaje
0,BAJO,668,33.3300
1,MEDIO,668,33.3300
2,ALTO,668,33.3300



=== TDNC POR CLASE ===


,count,mean,std,min,25%,50%,75%,max
TARGET_B,,,,,,,,
BAJO,668.0000,0.0134,0.0144,0.0000,0.0000,0.0082,0.0250,0.0462
MEDIO,668.0000,0.0949,0.0312,0.0463,0.0685,0.0924,0.1194,0.1572
ALTO,668.0000,0.3716,0.2132,0.1574,0.2115,0.3028,0.4581,1.1363



=== VERIFICACION VARIABLES DE EVENTO POR CLASE ===


,TASA_DESEMBOLSO,CLIENTES_PRESTAMO_EVENTO,MONTO_DESEMBOLSO_EVENTO,CLIENTES_NUEVOS_EVENTO
TARGET_B,,,,
BAJO,0.1585,9.9896,"108,976.3420",3.3445
MEDIO,0.0662,10.1089,"116,462.3129",2.6596
ALTO,0.0362,10.0178,"106,877.2508",3.0733


  CLIENTES_PRESTAMO_EVENTO: 152 NaN imputados con 0
  MONTO_DESEMBOLSO_EVENTO: 152 NaN imputados con 0
  CLIENTES_NUEVOS_EVENTO: 152 NaN imputados con 0

NaN restantes en variables de evento: 0

=== VERIFICACION FINAL VARIABLES DE EVENTO POR CLASE ===


,TASA_DESEMBOLSO,CLIENTES_PRESTAMO_EVENTO,MONTO_DESEMBOLSO_EVENTO,CLIENTES_NUEVOS_EVENTO
TARGET_B,,,,
BAJO,0.1585,9.8401,"107,344.9596",3.2945
MEDIO,0.0662,9.7911,"112,801.0725",2.5760
ALTO,0.0362,8.2032,"87,517.7488",2.5166



Interpretacion TASA_DESEMBOLSO:
  BAJO    : 0.1585 = 1 de cada 6 filas-grupo con desembolso
  MEDIO   : 0.0662 = 1 de cada 15 filas-grupo con desembolso
  ALTO    : 0.0362 = 1 de cada 28 filas-grupo con desembolso

Interpretacion TASA_DESEMBOLSO:
  BAJO    : 0.1585 = 1 de cada 6 filas-grupo con desembolso
  MEDIO   : 0.0662 = 1 de cada 15 filas-grupo con desembolso
  ALTO    : 0.0362 = 1 de cada 28 filas-grupo con desembolso


## 4. Split por asesor

In [4]:
def split_target_b(df_asesores, test_size=0.20, random_state=42):
    EXCLUIR_B = [
        'TARGET_B', 'TDNC', 'DIAS_MORA',
        'PRIMERA_OBS', 'ULTIMA_OBS',
        'ASESOR', 'NOMBRE', 'NUMERO_EMPLEADO',
        'PASE1', 'PASE15', 'PASE30',
        'DIAS_MORA_RANGO',
        # Leakage de construccion del target
        'ATRASO', 'PROVISION_HOY', 'PROVISION_SEMANT',
        'GASTO_PROVISION_SEMANAL', 'CARTERA_HOY', 'CARTERA_SEMANT',
    ]
    feature_cols = [c for c in df_asesores.columns if c not in EXCLUIR_B]

    df_clean = df_asesores.dropna(subset=['TARGET_B'])
    n_dropped = len(df_asesores) - len(df_clean)
    if n_dropped > 0:
        print(f'Asesores sin TARGET_B excluidos: {n_dropped}')

    X = df_clean[feature_cols]
    y = df_clean['TARGET_B'].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    assert len(set(X_train.index) & set(X_test.index)) == 0
    print('Anti-leakage OK: ningun asesor en train Y test simultaneamente')
    return X_train, X_test, y_train, y_test, feature_cols


print('=== SPLIT POR ASESOR ===')
X_train_B, X_test_B, y_train_B, y_test_B, features_B = split_target_b(df_asesores)

print(f'\nTrain: {len(X_train_B):,} asesores')
print(f'Test : {len(X_test_B):,} asesores')
print(f'Features: {len(features_B)}')
print('\nDistribucion train:')
print(y_train_B.value_counts().to_string())
print('\nDistribucion test:')
print(y_test_B.value_counts().to_string())
print('\nFeatures incluidas (variables de evento marcadas):')
vars_evento_nuevas = [
    'TASA_DESEMBOLSO', 'CLIENTES_PRESTAMO_EVENTO',
    'MONTO_DESEMBOLSO_EVENTO', 'CLIENTES_NUEVOS_EVENTO'
]
for f in sorted(features_B):
    marca = '<-- EVENTO (v3)' if f in vars_evento_nuevas else ''
    print(f'  {f} {marca}')


=== SPLIT POR ASESOR ===
Anti-leakage OK: ningun asesor en train Y test simultaneamente

Train: 1,603 asesores
Test : 401 asesores
Features: 29

Distribucion train:
TARGET_B
ALTO     535
BAJO     534
MEDIO    534

Distribucion test:
TARGET_B
BAJO     134
MEDIO    134
ALTO     133

Features incluidas (variables de evento marcadas):
  AREA 
  CLIENTES 
  CLIENTES_NUEVOS_EVENTO <-- EVENTO (v3)
  CLIENTES_PRESTAMO_EVENTO <-- EVENTO (v3)
  CODIGO_ASESOR 
  CODIGO_SUCURSAL 
  COORDINADOR 
  DIA_PAGO 
  EDAD 
  ESTADO_CIVIL 
  EXP_MICROFINANZAS 
  FECHA_ALTA 
  FECHA_BAJA 
  FECHA_NACIMIENTO 
  GRUPOS 
  INCREMENTO_CARTERA 
  MONTO_DESEMBOLSO_EVENTO <-- EVENTO (v3)
  MOTIVO_BAJA 
  NIVEL_ESTUDIOS 
  N_SEMANAS_OBS 
  PRODUCTO 
  PUESTO 
  RANGO_CICLO 
  REGION 
  SEXO 
  SUCURSAL 
  TASA_DESEMBOLSO <-- EVENTO (v3)
  TASA_PROM 
  TIPO_BAJA 


## 5. Guardar splits

In [5]:
df_train_B = X_train_B.copy()
df_train_B['TARGET_B'] = y_train_B.values
df_train_B['TDNC']     = df_asesores.loc[X_train_B.index, 'TDNC'].values

df_test_B = X_test_B.copy()
df_test_B['TARGET_B'] = y_test_B.values
df_test_B['TDNC']     = df_asesores.loc[X_test_B.index, 'TDNC'].values

df_train_B.to_parquet(INTERIM / 'target_b_train.parquet', index=False)
df_test_B.to_parquet(INTERIM  / 'target_b_test.parquet',  index=False)
df_asesores.to_parquet(INTERIM / 'target_b_full.parquet', index=False)

print(f'target_b_train.parquet -- {len(df_train_B):,} asesores')
print(f'target_b_test.parquet  -- {len(df_test_B):,} asesores')
print(f'target_b_full.parquet  -- {len(df_asesores):,} asesores')
print('\nVariables de evento correctamente calculadas e incluidas.')


target_b_train.parquet -- 1,603 asesores
target_b_test.parquet  -- 401 asesores
target_b_full.parquet  -- 2,004 asesores

Variables de evento correctamente calculadas e incluidas.


## 6. Registro formal de decisiones

In [6]:
print('''
=================================================================
  DECISIONES DE DISENO -- SPRINT 2 v3
=================================================================

  VARIABLES DE EVENTO REEMPLAZADAS (v3)

  Problema detectado:
  Las variables CLIENTES_PRESTAMO, PRESTAMO, CLIENTES_NUEVOS y
  DESEMBOLSO_CLIENTES_NUEVOS al promediarse sobre TODAS las semanas
  (incluyendo semanas sin desembolso) mezclan dos senales distintas:
  el tamano del grupo y la frecuencia de desembolso.

  Solucion:
  -> TASA_DESEMBOLSO = filas_evento / total_filas
     (mide actividad de renovacion de cartera)
  -> CLIENTES_PRESTAMO_EVENTO = mean(CLIENTES_PRESTAMO) donde > 0
     (tamano promedio real del grupo al desembolso)
  -> MONTO_DESEMBOLSO_EVENTO = mean(PRESTAMO) donde > 0
     (monto promedio real desembolsado)
  -> CLIENTES_NUEVOS_EVENTO = mean(CLIENTES_NUEVOS) donde > 0
     (clientes nuevos promedio en semanas de desembolso)

  TARGET: 3 CLASES (terciles de TDNC) -- sin cambios vs v2

  VARIABLES DE LEAKAGE EXCLUIDAS -- sin cambios
  -> PASE1, PASE15, PASE30, DIAS_MORA, DIAS_MORA_RANGO
  -> ATRASO, PROVISION_HOY, PROVISION_SEMANT
  -> GASTO_PROVISION_SEMANAL, CARTERA_HOY, CARTERA_SEMANT
=================================================================
''')



  DECISIONES DE DISENO -- SPRINT 2 v3

  VARIABLES DE EVENTO REEMPLAZADAS (v3)

  Problema detectado:
  Las variables CLIENTES_PRESTAMO, PRESTAMO, CLIENTES_NUEVOS y
  DESEMBOLSO_CLIENTES_NUEVOS al promediarse sobre TODAS las semanas
  (incluyendo semanas sin desembolso) mezclan dos senales distintas:
  el tamano del grupo y la frecuencia de desembolso.

  Solucion:
  -> TASA_DESEMBOLSO = filas_evento / total_filas
     (mide actividad de renovacion de cartera)
  -> CLIENTES_PRESTAMO_EVENTO = mean(CLIENTES_PRESTAMO) donde > 0
     (tamano promedio real del grupo al desembolso)
  -> MONTO_DESEMBOLSO_EVENTO = mean(PRESTAMO) donde > 0
     (monto promedio real desembolsado)
  -> CLIENTES_NUEVOS_EVENTO = mean(CLIENTES_NUEVOS) donde > 0
     (clientes nuevos promedio en semanas de desembolso)

  TARGET: 3 CLASES (terciles de TDNC) -- sin cambios vs v2

  VARIABLES DE LEAKAGE EXCLUIDAS -- sin cambios
  -> PASE1, PASE15, PASE30, DIAS_MORA, DIAS_MORA_RANGO
  -> ATRASO, PROVISION_HOY, PROVISION

In [7]:
# Identificar asesores sin ningun desembolso
sin_desembolso = vars_evento[vars_evento['TASA_DESEMBOLSO'] == 0].index
print(f"Asesores con TASA_DESEMBOLSO = 0: {len(sin_desembolso)}")

# Verificar su distribucion por TARGET_B
df_asesores['sin_desembolso'] = df_asesores['CODIGO_ASESOR'].isin(sin_desembolso)
print("\nDistribucion de asesores sin desembolso por clase:")
print(df_asesores.groupby('TARGET_B')['sin_desembolso'].sum().to_string())

# Ver cuantas semanas observadas tienen
print("\nSemanas observadas para asesores sin desembolso:")
print(df_asesores[df_asesores['sin_desembolso']][['N_SEMANAS_OBS','TDNC','TARGET_B']].describe().round(2))

Asesores con TASA_DESEMBOLSO = 0: 152

Distribucion de asesores sin desembolso por clase:
TARGET_B
BAJO      10
MEDIO     21
ALTO     121

Semanas observadas para asesores sin desembolso:
       N_SEMANAS_OBS     TDNC
count       152.0000 152.0000
mean         15.9100   0.4700
std          20.1400   0.3400
min           1.0000   0.0000
25%           3.0000   0.1900
50%           9.0000   0.4300
75%          20.0000   0.7000
max         114.0000   1.1400
